<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>1. Prod => for getting ML Score only for the past month (begin of monthly prod)</h1>
    <ul>
        <li>New column will be created => "Score ML"</li>
    </ul>
</div>

In [ ]:
# !pip install -r requirements.txt

In [1]:
from py_files import PARAMS_LOADING, PREPROCESSING, PREDICTION

Excel_launcher_path = r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\Launcher ML.xlsm"

params_principal, params_preprocessing, params_strat_selected, params_hyper_parameters = PARAMS_LOADING.get_all_params_from_excel(Excel_launcher_path)

Preprocessing Part

In [ ]:
# Create X's variations (change features, sector dummies) and Y's forard returns as target to predict (4 Minutes)
# Generate pickl "screen_ML_prod.pkl" as input for model's prediction in following steps
PREPROCESSING.preprocess_data(params_principal, params_preprocessing)

In [3]:
params_principal.loc['df_features_path']['param']

'\\\\groupe-ufg.com\\Commun\\Prive\\GestionAM\\Ingenierie_Financiere\\PROD\\_EQUITY\\1_FACTEUR_ML\\input_files\\screen_ML_prod.pkl'

Launch Prediction

In [4]:
import pandas as pd
# Step 1: Initialize parameters
input_transformed = pd.read_pickle(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\1_FACTEUR_ML\input_files\screen_ML_prod.pkl")
params = PARAMS_LOADING.unpack_strategy_parameters(params_strat_selected)

# Step 2: Prepare initial dataset, including create label columns, fill nan values and filter rows only in universe
screen_label = PREDICTION.labellize_data_and_fill_nan(input_transformed, params)

# Step 3: Calculate date ranges and filter data
data_train_and_test, date_ranges = PREDICTION.get_date_ranges(
    screen_label,
    params['training_window'],
    params['test_window']
)

# Step 4: Prepare test dataset
test_dataset = data_train_and_test.loc[
    data_train_and_test.index.get_level_values('Date') >= date_ranges['split_date']
].reset_index()

# Step 5: Generate predictions
# Split D-1
# Tester Tlt Secto ds le ML
y_pred, _, _, _, _, _ = PREDICTION.make_predictions(
    data_train_and_test,
    params,
    date_ranges['split_date'],
    params_hyper_parameters
)

# Step 6: Process predictions and calculate scores
output_last_month = PREDICTION.get_Score_ML_from_predictions(
    test_dataset,
    y_pred,
    params['period_to_predict']
)

In [5]:
output_last_month[["Date","Name",'Score ML']]

,Date,Name,Score ML
6591,2025-02-28,Societe Generale S.A. Class A,9.832496
6592,2025-02-28,Schneider Electric SE,5.862647
6593,2025-02-28,TotalEnergies SE,5.427136
6594,2025-02-28,Sanofi,5.142379
6595,2025-02-28,Airbus SE,7.236181
...,...,...,...
7183,2025-02-28,Carlsberg AS Class B,8.743719
7184,2025-02-28,Arkema SA,5.644891
7185,2025-02-28,BANK POLSKA KASA OPIEKI SA,2.747069
7186,2025-02-28,Vonovia SE,1.256281


<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>2. Adding ML Score column in screen_aggregate</h1>
</div>

In [6]:
import pandas as pd
screen_path = params_principal.loc['screen_path']['param']

In [7]:
# Adding only the last month Score Ml - 1m20s
update_score_ml = True
if update_score_ml:
    screen_agg = pd.read_pickle(screen_path)
    screen_agg.reset_index(inplace=True)
    screen_agg.set_index(['Company SEDOL','Date'],inplace=True)

    output_last_month.reset_index(inplace=True)
    output_last_month.set_index(['Company SEDOL','Date'],inplace=True)

    screen_agg.loc[output_last_month.index, 'Score ML'] = output_last_month['Score ML']

    screen_agg.reset_index(inplace=True)
    screen_agg.set_index('ISIN', inplace=True)

    screen_agg.to_pickle(screen_path)

In [29]:
screen_agg[screen_agg['Date'] == "2025-01-31"]['Score ML'].dropna()

ISIN
IT0004810054    6.415410
GB0031638363    6.599665
GB0003096442    3.735343
SE0000695876    6.633166
GB00B2B0DG97    7.747069
                  ...   
BE0003797140    8.626466
US35137L2043    8.844622
US35137L1052    8.266932
SE0006993770    3.366834
NO0010582521    6.331658
Name: Score ML, Length: 1099, dtype: float64

<div style="background-color: #013220; color: white; padding: 10px;">
    <h1>3. Générer Sec List</h1>
</div>

<div style="background-color: #013220; color: white; padding: 10px;">
    <h2>3.1 Générer Sec List avec Contrainte TE</h2>
</div>

In [1]:
import sys
sys.path.insert(0, r'\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\3_Sec_List_Backtest\\')
from backtest import backtest
from sec_list_generation import generic_histo_seclist, sec_list_ML_EU, worst_list_ML_EU, sec_list_spot, read_liste_noire

import datetime
from dateutil import relativedelta
import pandas as pd

In [2]:
# 30s loading
returns = pd.read_pickle(r"\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_RETURNS\returns.pkl") #Read the historical daily returns
screen_agg=pd.read_pickle(r"\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_SCREEN_AGG\screen_aggregate.pkl") #Read the screen of fundamental monthly variables
liste_noire = read_liste_noire([],[],r'\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_LISTE_NOIRE_EXCLUSION\Liste_Noire_Exclusion.xlsx')

In [3]:
#top_mandatory = ['Yes',10]
cut_score = 3
te_max = 0.02
max_weights = [0.025,0.015,0.02,0.02,0.02,0.02,0.02,0.03,0.015,0.02,0.02,0.035,0.03,0.02,0.02,0.02,0.02,0.02,0.02]
list_secto = [1,2,3,4,5,6,7,8,9,10,11,12,13,14,15,16,17,18,19]
critere = 'Score ML'
repechage_filter = ['Sector ICB19']
esg_cut = 0.2
nb_titres = 150
bench = 'STOXX EUROPE 600'
export_excel = True # If it's True, sec_list_ML_EU function won't return dataframe, but excel file locally for pushing on Bloomberg  # If ploting backtesting in notebook is needed, set it to TRUE
start_date = datetime.datetime(2010,2,1)
output_dir = r'\\groupe-ufg.com\Commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\0_PTF_BLOOM\PTF'
worst_percentile = 0.15

<div style="background-color: #013220; color: white; padding: 10px;">
3.1.1 Générer Sec List du Mois
</div>

In [4]:
# 5s execution
sec_list_ML_EU(screen_agg, output_dir, critere, max_weights, list_secto, returns, repechage_filter, liste_noire, nb_titres, te_max, esg_cut, bench, export_excel)

Found 10280 common stocks between screen and returns
After filtering for common stocks: 10280 stocks remaining


'ptf ML Europe generated successfully in \\\\groupe-ufg.com\\Commun\\Prive\\GestionAM\\Ingenierie_Financiere\\PROD\\_EQUITY\\0_PTF_BLOOM\\PTF/Pour March 2025/ML ESG EU.xlsx'

In [5]:
worst_list_ML_EU(screen_agg, output_dir, worst_percentile, critere, bench, export_excel)

'worst ML Europe generated successfully in \\\\groupe-ufg.com\\Commun\\Prive\\GestionAM\\Ingenierie_Financiere\\PROD\\_EQUITY\\0_PTF_BLOOM\\PTF/Pour March 2025/WORST ML EU.xlsx'

<div style="background-color: #013220; color: white; padding: 10px;">
3.1.2 Générer Sec List Historique
</div>

In [ ]:
# Generate Full Historical Securities List
full_histo = False
if full_histo:
    ptf = generic_histo_seclist(sec_list_ML_EU, start_date, screen_agg, output_dir, critere, max_weights, list_secto, returns, repechage_filter, liste_noire, nb_titres, te_max, esg_cut, bench, export_excel)
    ptf[['ISIN', 'Weight', 'Date']].to_pickle(r'\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\\Ptf histo\basket_histo.pkl')

In [ ]:
full_histo_worst = False
if full_histo_worst:
    ptf = generic_histo_seclist(worst_list_ML_EU, start_date, screen_agg, output_dir, worst_percentile, critere, bench, export_excel)
    ptf[['ISIN', 'Weight', 'Date']].to_pickle(r'\\groupe-ufg.com\commun\Prive\GestionAM\Ingenierie_Financiere\PROD\_EQUITY\\Ptf histo\worst_basket_histo.pkl')

<div style="background-color: #013220; color: white; padding: 10px;">
    <h2>3.2 Générer Sec List sans Contrainte</h2>
</div>

In [ ]:
metrics='Score ML'
percentile=0.15
bench='STOXX EUROPE 600'
output_dir=''
cut_mkt_cap=0
ptf_name='ML Q1'
score_neutral="ICB 19"
weight_neutral="ICB 19"
ponderation='Racine cube' # Racine cube, Racine carrée, Log, Equalweight
esg_exclusion=0.2
start_date = datetime.datetime(2010,2,1)

ptf = generic_histo_seclist(sec_list_spot, start_date, screen_agg, returns, bench, output_dir, percentile, cut_mkt_cap, metrics, ptf_name, score_neutral, weight_neutral, ponderation, esg_exclusion, liste_noire)

In [6]:
ptf = sec_list_spot(screen_agg, returns, bench, output_dir, percentile, cut_mkt_cap, metrics, ptf_name, score_neutral, weight_neutral, ponderation, esg_exclusion, liste_noire)

In [7]:
ptf

,PTF,ISIN,Weight,Date,Secto,Score
0,ML Q1,BMG7945E1057,0.000098,2005-01-01,12.0,1.000000
1,ML Q1,IT0005252140,0.000098,2005-01-01,12.0,1.000000
2,ML Q1,GB0007980591,0.000232,2005-01-01,12.0,1.000000
3,ML Q1,GB0007980591,0.000222,2005-01-01,12.0,1.000000
4,ML Q1,ES0178165017,0.000047,2005-01-01,12.0,1.000000
...,...,...,...,...,...,...
15821,ML Q1,CH0009002962,0.000097,2005-01-01,7.0,0.850797
15822,ML Q1,GB00B0LCW083,0.000086,2005-01-01,8.0,0.850781
15823,ML Q1,GB00BN7SWP63,0.000248,2005-01-01,8.0,0.850781
15824,ML Q1,GB00B0SWJX34,0.000033,2005-01-01,6.0,0.850765
